# Gold Price Time-Series Forecasting — LSTM (Google Colab)

**Pipeline:** cleaning is done **locally** (`clean_data.py` → `gold_clean.csv`).
This notebook handles the **GPU training** stage on Colab and exports the
artifacts that `app.py` loads for serving.

**Steps**
1. Setup & GPU check
2. Load the locally-cleaned data
3. Exploratory analysis
4. Train / test split (time-aware) + scaling
5. Build windowed sequences
6. Build & train the LSTM
7. Evaluate (MAE / RMSE / MAPE) & plot
8. Recursive multi-day forecast
9. Save artifacts (`model.keras`, `scaler.pkl`, `meta.json`) & download

> **Tip:** `Runtime → Change runtime type → T4 GPU` before running.

## 1. Setup & GPU check

In [ ]:
import numpy as np, pandas as pd, json, joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", gpus if gpus else "No GPU — using CPU (still works, just slower)")

## 2. Load the locally-cleaned data

Upload the `gold_clean.csv` produced by `clean_data.py`. The cell below tries a
Colab file upload first and falls back to a path if you've mounted Drive.

In [ ]:
CSV_PATH = "Gold Price (2013-2023).csv"
df = pd.read_csv(CSV_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
print("Shape:", df.shape)
print("Range:", df["Date"].min().date(), "->", df["Date"].max().date())
df.head()

In [ ]:
# Safety net: if you accidentally uploaded the RAW file, clean it inline so the
# notebook still runs. (Normally cleaning is done locally.)
if df.select_dtypes("object").shape[1] or "Vol." in df.columns:
    print("Raw file detected — applying inline cleaning.")
    raw = pd.read_csv(CSV_PATH)
    def _sn(s): return s.astype(str).str.replace(",","",regex=False).str.replace("%","",regex=False).str.strip()
    def _vol(s):
        s = s.astype(str).str.strip()
        mult = np.where(s.str.endswith("K"),1e3,np.where(s.str.endswith("M"),1e6,1.0))
        n = pd.to_numeric(s.str.replace("K","",regex=False).str.replace("M","",regex=False).replace("nan",np.nan),errors="coerce")
        return n*mult
    df = raw.copy()
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    for c in ["Price","Open","High","Low"]: df[c] = pd.to_numeric(_sn(df[c]),errors="coerce")
    df["Change %"] = pd.to_numeric(_sn(df["Change %"]),errors="coerce")
    df["Volume"] = _vol(df["Vol."]); df = df.drop(columns=["Vol."])
    df = df.sort_values("Date").reset_index(drop=True)
    df["Volume"] = df["Volume"].interpolate().bfill().ffill()
    df = df.dropna(subset=["Price"]).reset_index(drop=True)
print("Missing values:", int(df.isna().sum().sum()))
df.dtypes

## 3. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
ax[0].plot(df["Date"], df["Price"], lw=1, color="#C8961E")
ax[0].set_title("Gold Closing Price (USD/oz), 2013–2022"); ax[0].set_ylabel("Price")
ax[1].plot(df["Date"], df["Price"].rolling(30).mean(), label="30-day MA", color="#1f77b4")
ax[1].plot(df["Date"], df["Price"].rolling(90).mean(), label="90-day MA", color="#d62728")
ax[1].legend(); ax[1].set_title("Moving averages"); ax[1].set_ylabel("Price")
plt.tight_layout(); plt.show()

print(df["Price"].describe().round(2))

## 4. Train / test split (time-aware) & scaling

Time series must be split **chronologically** — no shuffling. We fit the scaler
**only on the training portion** to avoid look-ahead leakage.

In [ ]:
TARGET = "Price"
WINDOW = 60          # days of history fed to the model
TRAIN_FRAC = 0.80

series = df[TARGET].values.reshape(-1, 1)
split = int(len(series) * TRAIN_FRAC)

train_raw = series[:split]
test_raw  = series[split - WINDOW:]   # carry WINDOW days so test has full context

scaler = MinMaxScaler().fit(train_raw)
train_s = scaler.transform(train_raw)
test_s  = scaler.transform(test_raw)

print(f"Train days: {len(train_raw)} | Test days: {len(series) - split}")

## 5. Build windowed sequences

In [ ]:
def make_sequences(arr, window):
    X, y = [], []
    for i in range(window, len(arr)):
        X.append(arr[i - window:i, 0])
        y.append(arr[i, 0])
    return np.array(X)[..., None], np.array(y)

X_train, y_train = make_sequences(train_s, WINDOW)
X_test,  y_test  = make_sequences(test_s,  WINDOW)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## 6. Build & train the LSTM

In [ ]:
model = Sequential([
    Input((WINDOW, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.title("Training history"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend()
plt.show()

## 7. Evaluate on the held-out test set

In [ ]:
pred_s = model.predict(X_test, verbose=0)
pred = scaler.inverse_transform(pred_s)
true = scaler.inverse_transform(y_test.reshape(-1, 1))

mae  = mean_absolute_error(true, pred)
rmse = np.sqrt(mean_squared_error(true, pred))
mape = np.mean(np.abs((true - pred) / true)) * 100
print(f"MAE : {mae:8.2f} USD")
print(f"RMSE: {rmse:8.2f} USD")
print(f"MAPE: {mape:8.2f} %")

test_dates = df["Date"].values[split:]
plt.figure(figsize=(12, 5))
plt.plot(test_dates, true, label="Actual", color="#C8961E")
plt.plot(test_dates, pred, label="Predicted", color="#1f77b4", alpha=0.8)
plt.title("LSTM — Actual vs Predicted (test set)"); plt.ylabel("Price (USD/oz)")
plt.legend(); plt.tight_layout(); plt.show()

## 8. Recursive multi-day forecast

To forecast beyond the data we feed each prediction back in as the newest input
(autoregressive rollout).

In [ ]:
def forecast_future(model, scaler, recent_prices, window, horizon):
    """recent_prices: 1-D array of the most recent >=window raw prices."""
    scaled = scaler.transform(np.asarray(recent_prices).reshape(-1, 1)).ravel().tolist()
    out = []
    for _ in range(horizon):
        x = np.array(scaled[-window:]).reshape(1, window, 1)
        nxt = model.predict(x, verbose=0)[0, 0]
        out.append(nxt); scaled.append(nxt)
    return scaler.inverse_transform(np.array(out).reshape(-1, 1)).ravel()

HORIZON = 30
future = forecast_future(model, scaler, df[TARGET].values[-WINDOW:], WINDOW, HORIZON)
future_dates = pd.bdate_range(df["Date"].iloc[-1] + pd.Timedelta(days=1), periods=HORIZON)

plt.figure(figsize=(12, 5))
plt.plot(df["Date"].iloc[-180:], df[TARGET].iloc[-180:], label="History", color="#C8961E")
plt.plot(future_dates, future, label=f"{HORIZON}-day forecast", color="#d62728", ls="--", marker="o", ms=3)
plt.title("Gold price forecast"); plt.ylabel("Price (USD/oz)"); plt.legend()
plt.tight_layout(); plt.show()
pd.DataFrame({"Date": future_dates, "Forecast": future.round(2)}).head(10)

## 9. Save artifacts & download

In [ ]:
model.save("model.keras")
joblib.dump(scaler, "scaler.pkl")

meta = {
    "target": TARGET,
    "window": WINDOW,
    "train_frac": TRAIN_FRAC,
    "metrics": {"mae": float(mae), "rmse": float(rmse), "mape": float(mape)},
    "last_date": str(df["Date"].iloc[-1].date()),
    "last_window_prices": df[TARGET].values[-WINDOW:].round(2).tolist(),
}
with open("meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved: model.keras, scaler.pkl, meta.json")
try:
    from google.colab import files
    for f in ["model.keras", "scaler.pkl", "meta.json"]:
        files.download(f)
except Exception:
    print("Not on Colab — artifacts are in the working directory.")